# qwen3-tts
- Qwen 팀/Alibaba Cloud에서 공개한 TTS(Text-to-Speech) 모델 계열
- gTTS보다 좋은 점은 목소리 스타일 제어가 가능하다는 점
- 단점은 로컬에서 돌리면 GPU/VRAM이 필요할 수 있고, 모델 다운로드 용량도 있음

In [1]:
# qwen3 tts GitHub
# https://github.com/QwenLM/Qwen3-TTS

In [2]:
# uv add qwen-tts

## 1. 커스텀 보이스 생성

In [3]:
%pip install qwen_tts
import torch
from qwen_tts import Qwen3TTSModel

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")
print("qwen_tts import OK")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.1 MB/s eta 0:00:00
  Created wheel for sox: filename=sox-1.5.0-py3-none-any.whl size=40036 sha256=ce27891c91205a9d6


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 
사용 디바이스: cuda:0
qwen_tts import OK


### 1) 모델 불러오기
- Qwen3-TTS: Qwen 계열의 음성 합성 모델
- 12Hz: 음성을 토큰화하는 방식이 12Hz 기반
- 0.6B: 비교적 작은 경량 모델
- CustomVoice: 미리 제공된 화자 목소리를 선택하고, 지시문으로 말투를 조절하는 모델

In [4]:
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",         # 가장 가벼운 모델부터
    device_map=device,
    dtype=torch.float32,                            # CPU에서는 float32 권장
    attn_implementation="sdpa",                     # window는 sdpa 고정. 에러발생시 "eager"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.81G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

speech_tokenizer/model.safetensors:   0%|          | 0.00/682M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

### 2) 가능 목록 출력하기

In [5]:
# 지원 화자/언어 확인
print("화자 목록:", model.get_supported_speakers())
print("언어 목록:", model.get_supported_languages())

화자 목록: ['aiden', 'dylan', 'eric', 'ono_anna', 'ryan', 'serena', 'sohee', 'uncle_fu', 'vivian']
언어 목록: ['auto', 'chinese', 'english', 'french', 'german', 'italian', 'japanese', 'korean', 'portuguese', 'russian', 'spanish']


### 3) TTS

In [7]:
import soundfile as sf
import os # Import the os module

# 한국어 테스트
wavs, sr = model.generate_custom_voice(
    text="안녕하세요. 저는 인공지능 스피커입니다. 반가워요.",
    language="Korean",
    speaker="sohee"  # 목록 확인 후 한국어 지원 화자로 변경
)

# Create the directory if it doesn't exist
output_dir = "./audio/"
os.makedirs(output_dir, exist_ok=True)

sf.write(os.path.join(output_dir, "qwen3_test.wav"), wavs[0], sr)
print("생성완료")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


생성완료


In [8]:
from IPython.display import Audio, display

display(Audio("./audio/qwen3_test.wav", autoplay=True))

## 2. 보이스 디자인

In [9]:
model = Qwen3TTSModel.from_pretrained(
    'Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign',         # VoiceDesign 모델
    device_map=device,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
)

# single inference
wavs, sr = model.generate_voice_design(
    text='안녕하세요. 우중런 하고 싶었는데 비가 안 와서 아쉽습니다.',
    language='Korean',
    instruct='따뜻하고 친근한 20대 여성 목소리, 밝은 톤으로',           # 목소리 묘사
)
sf.write('voice_design.wav', wavs[0], sr)

display(Audio('voice_design.wav', autoplay=True))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


## 3. 보이스 클론

In [21]:
model = Qwen3TTSModel.from_pretrained(
    'Qwen/Qwen3-8B-Base',
    device_map=device,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
)

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.19G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 3.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.37 GiB is allocated by PyTorch, and 59.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
ref_audio = './audio/my_voice.mp3'
ref_text = '안녕하십니까. 반갑습니다.'

wavs, sr = model.generate_voice_clone(
    text='''교내 점심 추천 Streamlit 앱의 치명적인 오류로 인해 학생들이 학교 매점이 아닌 화성에서 점심을 먹는 기괴한 현상이 발생했습니다.
    급식 메뉴 대신 화면에 표기된 '테라포밍 특식 신청' 버튼을 누르자마자 교실 바닥이 붉은 토양의 화성 표면으로 전환되었습니다.
    학생들은 급식실 대신 우주복을 입은 AI 가이드에게 일일 산소 캡슐과 동결건조 배급량을 배정받았습니다.
    매점 빵을 기대했던 이들은 영하 60도의 대기 속에서 헬멧을 쓴 채 모래바람과 함께 식사를 마쳐야 했습니다.
    50분 만에 교실로 복귀한 학생들의 교복 주머니에서는 여전히 정체불명의 화성 규산염 먼지가 떨어지고 있습니다.''',
    language='Korean',
    ref_audio=ref_audio,
    ref_text=ref_text,
)
sf.write('voice_clone.wav', wavs[0], sr)

In [ ]:
display(Audio('voice_clone.wav', autoplay=True))